# GTSRB Research: Experiment 5 — Hybrid (AugMix + CBM) Robustness Benchmark

This notebook implements **Experiment 5 (Data-Centric + Representation-Centric Hybrid Robustness)** from our controlled research ladder:
$$\text{AugMix Triplet } (x_{clean}, x_{aug1}, x_{aug2}) \to \text{ResNet-18} \to \text{Concept Bottleneck } \hat{C} \in \mathbb{R}^{22} \to \text{Class Head } \hat{Y} \in \mathbb{R}^{43}$$

### Methodological Grounding:
- Tests **Hypothesis 3 ($H_3$ — Complementarity):** Does combining data-centric visual diversity (AugMix triplet consistency) with representation-centric semantic bottlenecks (CBM) produce complementary robustness improvements exceeding either individual intervention ($mCA_{Hybrid} > \max(mCA_{AugMix}, mCA_{CBM})$)?
- Dual-Level Consistency Regularization:
  1. **Task-Level JSD:** $\text{JSD}(\hat{y}_{clean}, \hat{y}_{aug1}, \hat{y}_{aug2})$ enforces invariant class predictions.
  2. **Concept-Level JSD:** $\frac{1}{4} \sum_{k} \text{JSD}(\hat{c}_{clean}^{(k)}, \hat{c}_{aug1}^{(k)}, \hat{c}_{aug2}^{(k)})$ enforces invariant physical concepts (Shape, Colour, Border, Category) across distortions.

### Experimental Controls (Strictly Identical to Experiments 1 through 4):
1. **Backbone:** ResNet-18 with ImageNet initialization and Zero-$\gamma$ residual BN initialization.
2. **Optimization:** SGD (Nesterov $\mu=0.9$, lr=0.01, weight decay $10^{-4}$ with No-Bias-Decay rule).
3. **Schedule:** 3-epoch linear warmup + Cosine Annealing decay over 15 epochs.
4. **Evaluation:** Clean accuracy, Macro-F1, ECE, and 15 held-out corruptions $\times$ 5 severities ($mCA$ and $RPD$).

In [ ]:
# --- Step 1: Environment Setup & GPU Verification ---
import os
from google.colab import drive
drive.mount('/content/drive')

# Check if dataset is already unzipped from prior runs
if not os.path.exists('/content/ML_Project/german'):
    print('Extracting base dataset from Drive...')
    !unzip -o -q "/content/drive/MyDrive/archive (1).zip" -d /content/ML_Project/german
else:
    print('Base dataset already present at /content/ML_Project!')

# Unpack Experiment 5 code package
if os.path.exists('/content/exp5_hybrid_code.zip'):
    print('Unpacking updated Experiment 5 code...')
    !unzip -o -q /content/exp5_hybrid_code.zip -d /content/ML_Project

%cd /content/ML_Project

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# --- Step 2: Triplet & Concept Consistency Sanity Check ---
import torch
from src.models.cbm import ConceptBottleneckModel, get_concept_lookup_tensors
from src.models.hybrid_loss import HybridCBMLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ConceptBottleneckModel(num_classes=43, pretrained=False, bottleneck_mode='residual').to(device)
model.eval()

# Simulate a batch of 2 triplets (clean, aug1, aug2) -> 6 images total
B = 2
x_clean = torch.randn(B, 3, 64, 64, device=device)
x_aug1 = torch.randn(B, 3, 64, 64, device=device)
x_aug2 = torch.randn(B, 3, 64, 64, device=device)
labels = torch.tensor([0, 14], dtype=torch.long, device=device)

shape_lut, colour_lut, border_lut, symcat_lut = get_concept_lookup_tensors(device)

with torch.no_grad():
    x_concat = torch.cat([x_clean, x_aug1, x_aug2], dim=0)
    out_concat = model(x_concat)
    out_clean = {k: v[:B] for k, v in out_concat.items()}
    out_aug1 = {k: v[B:2*B] for k, v in out_concat.items()}
    out_aug2 = {k: v[2*B:] for k, v in out_concat.items()}

criterion = HybridCBMLoss(lambda_c=1.0, lambda_aux=0.5, lambda_jsd_task=12.0, lambda_jsd_cpt=4.0)
loss, metrics = criterion(
    out_clean, out_aug1, out_aug2,
    labels, shape_lut[labels], colour_lut[labels], border_lut[labels], symcat_lut[labels]
)

print(f"Concatenated Batch Shape: {x_concat.shape}  (Expected: [6, 3, 64, 64])")
print(f"Total Hybrid Loss       : {loss.item():.4f}")
print(f"  - Task CE Loss        : {metrics['loss_task']:.4f}")
print(f"  - Concept CE Loss     : {metrics['loss_concepts']:.4f}")
print(f"  - Class JSD Loss      : {metrics['loss_jsd_task']:.4f}")
print(f"  - Concept JSD Loss    : {metrics['loss_jsd_cpt']:.4f}")
print("Hybrid Triplet & Consistency Forward Pass Verified Successfully!")

In [ ]:
# --- Step 3: Train Hybrid (AugMix + CBM) Model (Colab T4 GPU) ---
# 15 epochs takes ~10-12 minutes on T4 with Mixed Precision (AMP)
!python train_hybrid.py \
    --data-root german \
    --manifest-dir data/master_v1/metadata \
    --output-dir checkpoints/hybrid_resnet18 \
    --seed 42 \
    --epochs 15 \
    --warmup-epochs 3 \
    --batch-size 64 \
    --lr 0.01 \
    --lambda-c 1.0 \
    --lambda-aux 0.5 \
    --lambda-jsd-task 12.0 \
    --lambda-jsd-cpt 4.0 \
    --augmix-severity 3.0 \
    --bottleneck-mode residual

In [ ]:
# --- Step 4: Plot Multi-Scale Loss and Concept Validation Curves ---
import json
import pandas as pd
import matplotlib.pyplot as plt

with open('checkpoints/hybrid_resnet18/training_history.json', 'r') as f:
    history = json.load(f)

epochs_df = pd.DataFrame(history['epochs'])

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# 1. Multi-Scale Losses
ax1.plot(epochs_df['epoch'], epochs_df['train_loss'], label='Total Train Loss', color='black', lw=2)
ax1.plot(epochs_df['epoch'], epochs_df['train_loss_task'], label='Task CE', color='blue', lw=1.5, ls='--')
ax1.plot(epochs_df['epoch'], epochs_df['train_loss_jsd_task'], label='Task JSD', color='orange', lw=1.5)
ax1.plot(epochs_df['epoch'], epochs_df['train_loss_jsd_cpt'], label='Concept JSD', color='red', lw=1.5)
ax1.plot(epochs_df['epoch'], epochs_df['val_loss'], label='Val Loss', color='purple', lw=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Hybrid Multi-Scale Loss Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Sign Classification Performance
ax2.plot(epochs_df['epoch'], epochs_df['val_acc'] * 100, label='Val Accuracy (%)', color='green', lw=2)
ax2.plot(epochs_df['epoch'], epochs_df['val_macro_f1'] * 100, label='Val Macro-F1 (%)', color='navy', lw=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score (%)')
ax2.set_title('Classification Performance (Clean Canvas)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Concept Head Accuracies
ax3.plot(epochs_df['epoch'], epochs_df['val_acc_shape'] * 100, label='Shape Acc (%)', lw=1.8)
ax3.plot(epochs_df['epoch'], epochs_df['val_acc_colour'] * 100, label='Colour Acc (%)', lw=1.8)
ax3.plot(epochs_df['epoch'], epochs_df['val_acc_border'] * 100, label='Border Acc (%)', lw=1.8)
ax3.plot(epochs_df['epoch'], epochs_df['val_acc_symcat'] * 100, label='Symbol Cat Acc (%)', lw=1.8)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Concept Accuracy (%)')
ax3.set_title('Per-Axis Concept Accuracy')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Step 5: Clean Test & Robustness Benchmark (15 Corruptions x 5 Severities) ---
!python evaluate_robustness.py \
    --checkpoint checkpoints/hybrid_resnet18/best_hybrid_resnet18.pt \
    --data-root german \
    --meta-path data/master_v1/metadata/test.csv \
    --output-dir reports/hybrid_resnet18 \
    --batch-size 128

In [ ]:
# --- Step 6: 5-Way Comparative Analysis: Baseline vs AugMix vs Frequency vs CBM vs Hybrid ---
import json
from pathlib import Path
import pandas as pd

def load_summary(name, local_dir, drive_dir):
    p_local = Path(local_dir) / 'evaluation_summary.json'
    p_drive = Path(drive_dir) / 'evaluation_summary.json'
    if p_local.exists():
        with open(p_local) as f:
            return json.load(f)
    elif p_drive.exists():
        with open(p_drive) as f:
            return json.load(f)
    return None

models = [
    ('Baseline ResNet-18', 'reports/baseline_resnet18', '/content/drive/MyDrive/ML_Project_Results/baseline_resnet18'),
    ('AugMix ResNet-18', 'reports/augmix_resnet18', '/content/drive/MyDrive/ML_Project_Results/augmix_resnet18'),
    ('Frequency ResNet-18', 'reports/frequency_resnet18', '/content/drive/MyDrive/ML_Project_Results/frequency_resnet18'),
    ('CBM ResNet-18', 'reports/cbm_resnet18', '/content/drive/MyDrive/ML_Project_Results/cbm_resnet18'),
    ('Hybrid (AugMix+CBM)', 'reports/hybrid_resnet18', '/content/drive/MyDrive/ML_Project_Results/hybrid_resnet18'),
]

summaries = {}
for name, local_p, drive_p in models:
    s = load_summary(name, local_p, drive_p)
    if s is not None:
        summaries[name] = s

if 'Hybrid (AugMix+CBM)' in summaries:
    metrics = [
        ('Clean Accuracy', 'clean_accuracy', True),
        ('Clean Macro-F1', 'clean_macro_f1', True),
        ('Expected Calibration Error (ECE)', 'clean_ece', False),
        ('Mean Corruption Accuracy (mCA)', 'mean_corruption_accuracy', True),
        ('Relative Performance Drop (RPD)', 'relative_performance_drop', False),
    ]
    comp_records = []
    for title, key, is_pct in metrics:
        rec = {'Metric': title}
        for model_name, s in summaries.items():
            val = s.get(key, 0.0)
            rec[model_name] = f"{val*100:.2f}%" if is_pct else f"{val:.4f}"
        comp_records.append(rec)
    
    print('=' * 85)
    print('5-WAY EXPERIMENTAL BENCHMARK SUMMARY (GTSRB Held-Out + 75 Corruptions)')
    print('=' * 85)
    display(pd.DataFrame(comp_records))

    # Backup to Google Drive
    !mkdir -p "/content/drive/MyDrive/ML_Project_Results/hybrid_resnet18"
    !cp -r checkpoints/hybrid_resnet18/* "/content/drive/MyDrive/ML_Project_Results/hybrid_resnet18/"
    !cp -r reports/hybrid_resnet18/* "/content/drive/MyDrive/ML_Project_Results/hybrid_resnet18/"
    print("\nAll Hybrid Experiment 5 results backed up to Google Drive successfully!")
else:
    print('Hybrid evaluation summary not found yet. Make sure Step 5 completed successfully.')